# pSMAD — 02b_auto_roi_from_dapi

**Feeds:** Fig 1e, 1f

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 02b | Automated Cyst Segmentation + Scene Export (36 Locations)

This notebook is now the non-interactive preparation step for the pSMAD workflow.

It does four things:
- segments cysts from the DAPI channel of `well3-36locations.ome.tif`
- exports one BF / DAPI / pSMAD / bead TIFF per scene
- writes one cyst-label TIFF per scene
- writes the scene manifest and cyst-segmentation summary tables used downstream

It does **not** do any manual bead annotation. That is now handled in notebook `02c_manual_bead_well_annotation.ipynb`.

## Cell Guide

1. Imports.
2. Settings (paths/channels/segmentation).
3. Core helper functions for DAPI-based cyst segmentation.
4. Batch-run cyst segmentation across all 36 scenes and write scene exports + label masks + TSV outputs.
5. Render the full all-36 segmentation debug set, including any scene that fell back from the primary Otsu threshold.
6. Hand off to notebook `02c_manual_bead_well_annotation.ipynb` for manual bead-well annotation and ROI JSON writing.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from IPython.display import display
from skimage import color, filters, measure, morphology

In [ ]:
# -------------------------------
# Project + run settings
# -------------------------------
ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists() and (ROOT.parent / "scripts").exists():
    ROOT = ROOT.parent.resolve()

# Only this 36-location file is used for analysis in this workflow.
LOCATIONS_OME = ROOT / "results/converted_tiff/well3-36locations.ome.tif"
assert LOCATIONS_OME.exists(), f"Missing file: {LOCATIONS_OME}"

OUT_BASE_DIR = ROOT / "results/manual_bead_well_36locations"
OUT_SCENE_DIR = OUT_BASE_DIR / "scene_images"
OUT_LABEL_DIR = OUT_BASE_DIR / "labels"
OUT_TABLE_DIR = ROOT / "results/tables"

SCENE_MANIFEST_TSV = OUT_TABLE_DIR / "scene_manifest_36locations.tsv"
CYST_STATS_TSV = OUT_TABLE_DIR / "cyst_segmentation_stats_36locations.tsv"

# Scene-level channel mapping in well3-36locations.
# 0=BF, 1=DAPI, 2=pSMAD(568), 3=bead(647)
CHANNEL_MAP = {
    "brightfield": 0,
    "dapi": 1,
    "psmad": 2,
    "bead": 3,
}

PROJECTION = "max"

# Expected cyst count defaults + scene-specific overrides from manual review.
EXPECTED_CYSTS_PER_SCENE = 7
EXPECTED_CYSTS_BY_SCENE = {
    4: 6,
    7: 6,
    31: 6,
    33: 6,
    35: 6,
}

# Cyst segmentation parameters. Thresholding is intentionally Otsu-only.
CYST_PARAMS = {
    "baseline_percentile": 5.0,
    "threshold_method": "otsu",
    "threshold_percentile": 95.0,
    "threshold_scale": 1.00,
    "threshold_offset": 0.0,
    "multi_otsu_classes": 3,
    "min_size_px": 40,
    "closing_disk_px": 3,
    "dilation_disk_px": 3,
    "hole_area_px": 80,
    "area_min_px": 500,
    "area_max_px": 300000,
    "eccentricity_max": 0.95,
    "solidity_min": 0.68,
    "border_margin_px": 10,
    "target_count": EXPECTED_CYSTS_PER_SCENE,
    "auto_tune_if_count_mismatch": True,
    "auto_percentiles": [90, 92, 94, 95, 96, 97, 98],
    "auto_scales": [0.80, 0.90, 1.00, 1.10, 1.20, 1.30],
    "auto_max_trials": 220,
    "trim_to_target_if_over": True,
}

REQUIRE_36_SCENES = True
SCENE_FOR_DEBUG = 6
SCENE_OVERLAY_DOWNSAMPLE = 2
SCENE_OVERLAY_GRID_COLS = 6

# Optimization mode suppresses notebook displays/figures but still writes core outputs.
OPTIMIZATION_MODE = True
RENDER_SEGMENTATION_QA_MONTAGE = not OPTIMIZATION_MODE
RENDER_ALL_SCENE_SEGMENTATION_DEBUG = not OPTIMIZATION_MODE

EXPORT_SCENE_CHANNEL_TIFFS = True
OVERWRITE_SCENE_EXPORTS = True

for p in [OUT_BASE_DIR, OUT_SCENE_DIR, OUT_LABEL_DIR, OUT_TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Input OME-TIFF:", LOCATIONS_OME)
print("Scene manifest TSV:", SCENE_MANIFEST_TSV)
print("Cyst stats TSV:", CYST_STATS_TSV)

Project root: <analysis-root>/pSMAD
Input OME-TIFF: <analysis-root>/pSMAD/results/converted_tiff/well3-36locations.ome.tif
Scene manifest TSV: <analysis-root>/pSMAD/results/tables/scene_manifest_36locations.tsv
Cyst stats TSV: <analysis-root>/pSMAD/results/tables/cyst_segmentation_stats_36locations.tsv


## Settings Notes

- `CYST_PARAMS` is intentionally kept close to the prior working notebook so cyst detection behavior stays consistent while we improve downstream quantification.
- This notebook is locked to `well3-36locations.ome.tif` only.
- Otsu is hard-coded as the thresholding method. If the component count mismatches, the notebook first trims obvious extra fragments and only then explores Otsu-scaled variants.
- The outputs from this notebook are the only inputs needed by manual bead annotation notebook `02c`.

In [ ]:
# -------------------------------
# Core helper functions
# -------------------------------
def collapse_to_2d(array: np.ndarray, projection: str = "max") -> np.ndarray:
    """Collapse non-spatial dimensions and return one 2D image."""
    array = np.asarray(array)
    if array.ndim == 2:
        return array.astype(np.float32)
    planes = array.reshape((-1, array.shape[-2], array.shape[-1]))
    if projection == "max":
        out = planes.max(axis=0)
    elif projection == "mean":
        out = planes.mean(axis=0)
    elif projection == "first":
        out = planes[0]
    else:
        raise ValueError(f"Unknown projection mode: {projection}")
    return out.astype(np.float32)


def load_scene_channels(
    ome_path: Path,
    scene_index: int,
    channel_map: dict[str, int],
    projection: str = "max",
) -> tuple[dict[str, np.ndarray], tuple[int, int], tuple[int, int], str]:
    """Load one scene and split channels by configured indices."""
    with tifffile.TiffFile(ome_path) as tif:
        series0_shape = tuple(int(x) for x in tif.series[0].shape[-2:])
        series = tif.series[scene_index]
        scene_shape = tuple(int(x) for x in series.shape[-2:])
        scene_name = str(getattr(series, "name", f"series_{scene_index:02d}"))
        arr = np.asarray(series.asarray())
        axes = series.axes

    if "C" not in axes:
        raise ValueError(
            f"Scene {scene_index} has no channel axis. axes={axes}, shape={arr.shape}"
        )

    c_axis = axes.index("C")
    channels: dict[str, np.ndarray] = {}
    for name, c_idx in channel_map.items():
        if c_idx >= arr.shape[c_axis]:
            raise IndexError(
                f"Channel {c_idx} out of bounds for scene {scene_index} (shape={arr.shape}, axes={axes})"
            )
        ch = np.take(arr, indices=c_idx, axis=c_axis)
        channels[name] = collapse_to_2d(ch, projection=projection)

    return channels, scene_shape, series0_shape, scene_name


def read_ome_physical_pixel_size_um(ome_path: Path) -> float:
    """Read the physical XY pixel size directly from OME metadata in microns."""
    with tifffile.TiffFile(ome_path) as tif:
        omexml = tif.ome_metadata or ""

    import re

    m_x = re.search(r'PhysicalSizeX="([^"]+)"', omexml)
    m_y = re.search(r'PhysicalSizeY="([^"]+)"', omexml)
    if m_x is None or m_y is None:
        raise RuntimeError(
            f"OME metadata missing PhysicalSizeX/PhysicalSizeY in {ome_path}"
        )

    px_x = float(m_x.group(1))
    px_y = float(m_y.group(1))
    if not np.isfinite(px_x) or not np.isfinite(px_y) or px_x <= 0 or px_y <= 0:
        raise RuntimeError(
            f"Invalid OME physical pixel size in {ome_path}: x={px_x}, y={px_y}"
        )
    if not np.isclose(px_x, px_y, rtol=1e-6, atol=1e-9):
        raise RuntimeError(
            f"Anisotropic OME pixel size not supported here: x={px_x}, y={px_y}"
        )
    return float(px_x)


def scaled_pixel_size_um(
    base_pixel_um: float,
    series0_shape_yx: tuple[int, int],
    scene_shape_yx: tuple[int, int],
) -> float:
    """Scale physical pixel size if scene pixels are resized relative to the source series."""
    sy = float(series0_shape_yx[0]) / float(scene_shape_yx[0])
    sx = float(series0_shape_yx[1]) / float(scene_shape_yx[1])
    return float(base_pixel_um * ((sx + sy) * 0.5))


def normalize_for_display(
    img: np.ndarray, p_lo: float = 2, p_hi: float = 99.8
) -> np.ndarray:
    """Percentile normalization for plotting only."""
    finite = img[np.isfinite(img)]
    lo, hi = np.percentile(finite, [p_lo, p_hi])
    den = max(1e-6, float(hi - lo))
    return np.clip((img - lo) / den, 0, 1)


def expected_cyst_count_for_scene(scene_index: int) -> int:
    """Scene-specific expected cyst count (default + overrides)."""
    return int(EXPECTED_CYSTS_BY_SCENE.get(int(scene_index), EXPECTED_CYSTS_PER_SCENE))


def compute_cyst_threshold(flat: np.ndarray, params: dict) -> tuple[float, dict]:
    """Compute the Otsu-derived cyst threshold and record only Otsu diagnostics."""
    finite = flat[np.isfinite(flat)]
    if finite.size == 0:
        raise ValueError("Flattened DAPI has no finite pixels.")

    method = str(params.get("threshold_method", "otsu")).lower()
    if method != "otsu":
        raise ValueError(
            "This workflow now hard-codes Otsu thresholding. Set threshold_method='otsu'."
        )

    raw_threshold = float(filters.threshold_otsu(finite))
    if not np.isfinite(raw_threshold):
        raise ValueError("Otsu threshold produced a non-finite value.")

    scale = float(params.get("threshold_scale", 1.0))
    offset = float(params.get("threshold_offset", 0.0))
    threshold_value = float(raw_threshold * scale + offset)

    debug = {
        "method": method,
        "raw_threshold": raw_threshold,
        "threshold_scale": scale,
        "threshold_offset": offset,
        "threshold_value": threshold_value,
        "candidate_thresholds": {"otsu": float(raw_threshold)},
    }
    return threshold_value, debug


def segment_cysts_from_dapi(
    dapi: np.ndarray, params: dict
) -> tuple[np.ndarray, list, dict[str, np.ndarray]]:
    """Segment cyst masks from DAPI. Auto-tune threshold to match expected count."""
    baseline = float(np.percentile(dapi, params["baseline_percentile"]))
    flat = np.clip(dapi - baseline, a_min=0.0, a_max=None)
    h, w = dapi.shape

    def run_single_threshold(thr_value: float):
        mask_thr = flat > float(thr_value)
        mask = morphology.remove_small_objects(
            mask_thr, min_size=int(params["min_size_px"])
        )
        if params["closing_disk_px"] > 0:
            mask = morphology.binary_closing(
                mask, morphology.disk(int(params["closing_disk_px"]))
            )
        if params["dilation_disk_px"] > 0:
            mask = morphology.binary_dilation(
                mask, morphology.disk(int(params["dilation_disk_px"]))
            )
        if params["hole_area_px"] > 0:
            mask = morphology.remove_small_holes(
                mask, area_threshold=int(params["hole_area_px"])
            )

        labels_raw = measure.label(mask)
        props_raw = measure.regionprops(labels_raw)

        labels = np.zeros_like(labels_raw, dtype=np.int32)
        keep_count = 0
        margin = int(params["border_margin_px"])

        for region in props_raw:
            if (
                region.area < params["area_min_px"]
                or region.area > params["area_max_px"]
            ):
                continue
            if region.eccentricity > params["eccentricity_max"]:
                continue
            if region.solidity < params["solidity_min"]:
                continue

            minr, minc, maxr, maxc = region.bbox
            if (
                minr <= margin
                or minc <= margin
                or maxr >= (h - margin)
                or maxc >= (w - margin)
            ):
                continue

            keep_count += 1
            labels[labels_raw == region.label] = keep_count

        props = measure.regionprops(labels)
        return (
            mask_thr.astype(np.uint8),
            mask.astype(np.uint8),
            labels_raw.astype(np.int32),
            labels.astype(np.int32),
            props,
            int(len(props_raw)),
        )

    thr, threshold_info = compute_cyst_threshold(flat, params)
    mask_thr, mask_clean, labels_raw, labels, props, n_raw_components = (
        run_single_threshold(thr)
    )

    selected_variant = "primary"
    autotune_used = False
    autotune_rows: list[dict] = []

    target_count_raw = params.get("target_count", None)
    target_count = int(target_count_raw) if target_count_raw is not None else None

    def trim_to_target(labels_in: np.ndarray, props_in: list, target_n: int):
        props_sorted = sorted(props_in, key=lambda r: float(r.area), reverse=True)[
            :target_n
        ]
        labels_trim = np.zeros_like(labels_in, dtype=np.int32)
        for i, region in enumerate(props_sorted, start=1):
            labels_trim[labels_in == region.label] = i
        return labels_trim, measure.regionprops(labels_trim)

    trim_applied = False
    if (
        target_count is not None
        and bool(params.get("trim_to_target_if_over", True))
        and len(props) > target_count
    ):
        labels, props = trim_to_target(labels, props, target_count)
        trim_applied = True

    if target_count is not None and bool(
        params.get("auto_tune_if_count_mismatch", False)
    ):
        if len(props) != target_count:
            raw_otsu = float(threshold_info["raw_threshold"])
            offset = float(threshold_info.get("threshold_offset", 0.0))
            scales = [float(s) for s in params.get("auto_scales", [1.0])]
            max_trials = int(params.get("auto_max_trials", 120))

            def score_candidate(
                n_components: int,
                areas: list[float],
                scale_deviation: float,
            ) -> tuple:
                count_error = abs(n_components - target_count)
                under_penalty = 1 if n_components < target_count else 0
                tiny_penalty = int(
                    sum(a < (params["area_min_px"] * 1.5) for a in areas)
                )
                median_area = float(np.median(areas)) if len(areas) > 0 else 0.0
                return (
                    count_error,
                    under_penalty,
                    float(scale_deviation),
                    tiny_penalty,
                    -median_area,
                )

            current_areas = [float(r.area) for r in props]
            best_score = score_candidate(len(props), current_areas, 0.0)
            best_bundle = None

            trial_idx = 0
            for scale in scales:
                trial_idx += 1
                if trial_idx > max_trials:
                    break

                thr_test = max(0.0, raw_otsu * float(scale) + offset)
                (
                    mask_thr_t,
                    mask_clean_t,
                    labels_raw_t,
                    labels_t,
                    props_t,
                    n_raw_t,
                ) = run_single_threshold(thr_test)

                trim_t = False
                if (
                    target_count is not None
                    and bool(params.get("trim_to_target_if_over", True))
                    and len(props_t) > target_count
                ):
                    labels_t, props_t = trim_to_target(labels_t, props_t, target_count)
                    trim_t = True

                areas_t = [float(r.area) for r in props_t]
                score_t = score_candidate(
                    len(props_t),
                    areas_t,
                    abs(float(scale) - 1.0),
                )

                autotune_rows.append(
                    {
                        "variant": f"otsu x{scale:.2f}",
                        "threshold": float(thr_test),
                        "n_components": int(len(props_t)),
                        "count_error": int(abs(len(props_t) - target_count)),
                        "median_area_px": (
                            float(np.median(areas_t)) if len(areas_t) > 0 else 0.0
                        ),
                        "trim_applied": bool(trim_t),
                    }
                )

                if score_t < best_score:
                    best_score = score_t
                    best_bundle = {
                        "variant": f"otsu x{scale:.2f}",
                        "threshold": float(thr_test),
                        "mask_thr": mask_thr_t,
                        "mask_clean": mask_clean_t,
                        "labels_raw": labels_raw_t,
                        "labels": labels_t,
                        "props": props_t,
                        "n_raw": int(n_raw_t),
                        "trim_applied": bool(trim_t),
                    }

            if best_bundle is not None:
                selected_variant = str(best_bundle["variant"])
                thr = float(best_bundle["threshold"])
                mask_thr = best_bundle["mask_thr"]
                mask_clean = best_bundle["mask_clean"]
                labels_raw = best_bundle["labels_raw"]
                labels = best_bundle["labels"]
                props = best_bundle["props"]
                n_raw_components = int(best_bundle["n_raw"])
                trim_applied = bool(best_bundle.get("trim_applied", trim_applied))
                autotune_used = True

    threshold_info = dict(threshold_info)
    threshold_info["selected_variant"] = selected_variant
    threshold_info["threshold_value"] = float(thr)

    debug = {
        "baseline_value": np.float32(baseline),
        "flat": flat.astype(np.float32),
        "mask_threshold": mask_thr.astype(np.uint8),
        "mask_clean": mask_clean.astype(np.uint8),
        "labels_raw": labels_raw.astype(np.int32),
        "threshold_info": threshold_info,
        "n_raw_components": int(n_raw_components),
        "n_kept_components": int(len(props)),
        "autotune_used": bool(autotune_used),
        "autotune_rows": autotune_rows,
        "autotune_target_count": target_count,
        "trim_applied": bool(trim_applied),
    }
    return labels, props, debug


def mask_to_polygon_xy(
    mask: np.ndarray, simplify_tolerance_px: float = 1.5, min_points: int = 8
) -> list[list[float]]:
    """Convert a binary mask to [x, y] polygon vertices."""
    contours = measure.find_contours(mask.astype(np.uint8), 0.5)
    if not contours:
        return []
    contour = max(contours, key=lambda c: c.shape[0])
    if simplify_tolerance_px > 0:
        contour = measure.approximate_polygon(contour, tolerance=simplify_tolerance_px)
    pts = [[float(col), float(row)] for row, col in contour]
    if len(pts) < min_points:
        return []
    return pts


def rel_to_root(path: Path) -> str:
    """Store paths relative to project root when possible."""
    try:
        return str(path.resolve().relative_to(ROOT.resolve()))
    except Exception:
        return str(path.resolve())


def plot_cyst_segmentation_debug_figure(
    channels: dict[str, np.ndarray],
    cyst_labels: np.ndarray,
    cyst_props: list,
    cyst_debug: dict,
    scene_index: int,
    scene_name: str,
    expected_cysts: int,
) -> None:
    """Render the standard 8-panel segmentation debug figure for one scene."""
    thr_dbg = cyst_debug["threshold_info"]
    selected_variant = str(thr_dbg.get("selected_variant", "primary"))
    used_alternate = selected_variant != "primary"

    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    axes = axes.ravel()

    axes[0].imshow(normalize_for_display(channels["brightfield"]), cmap="gray")
    axes[0].set_title("Brightfield")
    axes[0].axis("off")

    axes[1].imshow(normalize_for_display(channels["dapi"]), cmap="gray")
    axes[1].set_title("DAPI raw")
    axes[1].axis("off")

    axes[2].imshow(normalize_for_display(cyst_debug["flat"]), cmap="gray")
    axes[2].set_title("Flattened DAPI (raw - baseline)")
    axes[2].axis("off")

    axes[3].imshow(cyst_debug["mask_threshold"], cmap="gray")
    axes[3].set_title(f"Threshold mask (thr={thr_dbg['threshold_value']:.1f})")
    axes[3].axis("off")

    axes[4].imshow(cyst_debug["mask_clean"], cmap="gray")
    axes[4].set_title("Mask after morphology")
    axes[4].axis("off")

    raw_overlay = color.label2rgb(
        cyst_debug["labels_raw"],
        image=normalize_for_display(channels["dapi"]),
        bg_label=0,
        alpha=0.35,
    )
    axes[5].imshow(np.clip(raw_overlay, 0.0, 1.0))
    axes[5].set_title("Raw connected components")
    axes[5].axis("off")

    final_overlay = color.label2rgb(
        cyst_labels,
        image=normalize_for_display(channels["dapi"]),
        bg_label=0,
        alpha=0.35,
    )
    axes[6].imshow(np.clip(final_overlay, 0.0, 1.0))
    axes[6].set_title(f"Filtered cyst labels (n={len(cyst_props)})")
    axes[6].axis("off")

    flat_vals = cyst_debug["flat"].ravel()
    flat_vals = flat_vals[np.isfinite(flat_vals)]
    axes[7].hist(flat_vals, bins=256, color="#808080")
    legend_handles = []
    legend_labels = []

    otsu_value = float(thr_dbg["candidate_thresholds"].get("otsu", np.nan))
    if np.isfinite(otsu_value):
        otsu_handle = axes[7].axvline(
            otsu_value, color="#f58518", linestyle="--", linewidth=1.4
        )
        legend_handles.append(otsu_handle)
        legend_labels.append("otsu")

    selected_handle = axes[7].axvline(
        float(thr_dbg["threshold_value"]), color="red", linewidth=2.2
    )
    legend_handles.append(selected_handle)
    legend_labels.append(f"selected: {selected_variant}")
    axes[7].set_title("Flat DAPI histogram (Otsu only)")
    axes[7].set_yscale("log")
    axes[7].legend(legend_handles, legend_labels, fontsize=7, loc="upper right")

    summary = (
        f"S{scene_index:02d} ({scene_name}) | cysts={len(cyst_props)}/{expected_cysts} | "
        f"method={thr_dbg['method']} | variant={selected_variant} | thr={thr_dbg['threshold_value']:.1f}"
    )
    if used_alternate:
        summary = "FALLBACK FROM PRIMARY OTSU | " + summary
    fig.suptitle(
        summary, fontsize=12, color=("red" if used_alternate else "black"), y=0.98
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

## All-Scene Cyst Segmentation Diagnostics

The next code cell renders the same 8-panel segmentation debug figure for **all 36 scenes**. Any scene that fell back from the primary Otsu threshold is flagged in **red**.

In [ ]:
# -------------------------------
# Batch cyst segmentation across all scenes (36 expected)
# -------------------------------
with tifffile.TiffFile(LOCATIONS_OME) as tif:
    n_scenes_total = len(tif.series)

if REQUIRE_36_SCENES and n_scenes_total != 36:
    raise RuntimeError(
        f"Expected 36 scenes in {LOCATIONS_OME.name}, found {n_scenes_total}."
    )

scene_rows = []
stats_rows = []
cyst_overlay_previews = []
base_pixel_um = read_ome_physical_pixel_size_um(LOCATIONS_OME)
print(f"OME physical pixel size: {base_pixel_um:.4f} um/px")

for scene_idx in range(n_scenes_total):
    channels, scene_shape, series0_shape, scene_name = load_scene_channels(
        LOCATIONS_OME,
        scene_idx,
        CHANNEL_MAP,
        projection=PROJECTION,
    )
    pixel_um = scaled_pixel_size_um(base_pixel_um, series0_shape, scene_shape)

    expected_cysts = expected_cyst_count_for_scene(scene_idx)
    cyst_params = dict(CYST_PARAMS)
    cyst_params["target_count"] = int(expected_cysts)
    cyst_labels, cyst_props, cyst_debug = segment_cysts_from_dapi(
        channels["dapi"], cyst_params
    )

    scene_key = f"scene{scene_idx:02d}"
    bf_path = OUT_SCENE_DIR / f"{scene_key}_bf.tif"
    dapi_path = OUT_SCENE_DIR / f"{scene_key}_dapi.tif"
    psmad_path = OUT_SCENE_DIR / f"{scene_key}_psmad.tif"
    bead_path = OUT_SCENE_DIR / f"{scene_key}_bead647.tif"
    cyst_labels_path = OUT_LABEL_DIR / f"{scene_key}_cyst_labels.tif"

    if EXPORT_SCENE_CHANNEL_TIFFS:
        if OVERWRITE_SCENE_EXPORTS or not bf_path.exists():
            tifffile.imwrite(bf_path, channels["brightfield"].astype(np.float32))
        if OVERWRITE_SCENE_EXPORTS or not dapi_path.exists():
            tifffile.imwrite(dapi_path, channels["dapi"].astype(np.float32))
        if OVERWRITE_SCENE_EXPORTS or not psmad_path.exists():
            tifffile.imwrite(psmad_path, channels["psmad"].astype(np.float32))
        if OVERWRITE_SCENE_EXPORTS or not bead_path.exists():
            tifffile.imwrite(bead_path, channels["bead"].astype(np.float32))
        if OVERWRITE_SCENE_EXPORTS or not cyst_labels_path.exists():
            tifffile.imwrite(cyst_labels_path, cyst_labels.astype(np.uint16))

    cyst_count = int(len(cyst_props))
    cyst_count_ok = bool(cyst_count == expected_cysts)
    thr_info = cyst_debug["threshold_info"]

    scene_row = {
        "source_file": LOCATIONS_OME.name,
        "source_ome_path": rel_to_root(LOCATIONS_OME),
        "scene_index": int(scene_idx),
        "scene_name": str(scene_name),
        "scene": f"series_{scene_idx:02d}",
        "brightfield_image_path": rel_to_root(bf_path),
        "dapi_image_path": rel_to_root(dapi_path),
        "psmad_image_path": rel_to_root(psmad_path),
        "bead_image_path": rel_to_root(bead_path),
        "cyst_labels_path": rel_to_root(cyst_labels_path),
        "pixel_size_um": float(pixel_um),
        "n_cysts_detected": int(cyst_count),
        "expected_cysts": int(expected_cysts),
    }
    scene_rows.append(scene_row)

    stats_rows.append(
        {
            "scene_index": int(scene_idx),
            "scene_name": str(scene_name),
            "n_cysts_detected": int(cyst_count),
            "expected_cysts": int(expected_cysts),
            "cyst_count_ok": bool(cyst_count_ok),
            "threshold_method": str(thr_info["method"]),
            "threshold_variant": str(thr_info.get("selected_variant", "primary")),
            "threshold_value": float(thr_info["threshold_value"]),
            "autotune_used": bool(cyst_debug.get("autotune_used", False)),
            "trim_applied": bool(cyst_debug.get("trim_applied", False)),
        }
    )

    # Build a downsampled cyst overlay preview for quick all-scene QA in notebook output.
    overlay = color.label2rgb(
        cyst_labels,
        image=normalize_for_display(channels["brightfield"]),
        bg_label=0,
        alpha=0.35,
    )
    ds = max(1, int(SCENE_OVERLAY_DOWNSAMPLE))
    preview = (np.clip(overlay[::ds, ::ds], 0.0, 1.0) * 255).astype(np.uint8)
    cyst_overlay_previews.append(
        {
            "scene_index": int(scene_idx),
            "n_cysts": int(cyst_count),
            "expected_cysts": int(expected_cysts),
            "ok": bool(cyst_count_ok),
            "image": preview,
        }
    )

    status = "OK" if cyst_count_ok else "FLAG"
    print(
        f"[{status}] scene {scene_idx:02d} | cysts={cyst_count} (expected {expected_cysts})"
    )

scene_manifest_df = (
    pd.DataFrame(scene_rows).sort_values("scene_index").reset_index(drop=True)
)
scene_stats_df = (
    pd.DataFrame(stats_rows).sort_values("scene_index").reset_index(drop=True)
)

scene_manifest_df.to_csv(SCENE_MANIFEST_TSV, sep="	", index=False)
scene_stats_df.to_csv(CYST_STATS_TSV, sep="	", index=False)

print("---")
print("Scene manifest TSV:", SCENE_MANIFEST_TSV)
print("Cyst stats TSV:", CYST_STATS_TSV)
print("Scenes processed:", len(scene_manifest_df))
print("Scenes with expected cyst count:", int(scene_stats_df["cyst_count_ok"].sum()))
print("Flagged scenes:", int((~scene_stats_df["cyst_count_ok"]).sum()))

if OPTIMIZATION_MODE:
    print("Skipping scene stats displays (OPTIMIZATION_MODE=True).")
else:
    display(scene_stats_df)

    print("Cyst-count distribution across scenes:")
    display(
        scene_stats_df["n_cysts_detected"]
        .value_counts()
        .sort_index()
        .rename_axis("n_cysts")
        .reset_index(name="n_scenes")
    )

# Show all-scene cyst overlay montage (before manual bead annotation) unless disabled for optimization.
if RENDER_SEGMENTATION_QA_MONTAGE:
    sorted_previews = sorted(cyst_overlay_previews, key=lambda d: d["scene_index"])
    n = len(sorted_previews)
    ncols = max(1, int(SCENE_OVERLAY_GRID_COLS))
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.0 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, item in zip(axes, sorted_previews):
        ax.imshow(item["image"])
        title_color = "black" if item["ok"] else "red"
        ax.set_title(
            f"S{item['scene_index']:02d} | c={item['n_cysts']}/{item['expected_cysts']}",
            fontsize=8,
            color=title_color,
        )
        ax.axis("off")

    for ax in axes[len(sorted_previews) :]:
        ax.axis("off")

    fig.suptitle("Cyst segmentation QA montage (all 36 scenes)", fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print(
        "Skipping all-scene cyst segmentation QA montage (RENDER_SEGMENTATION_QA_MONTAGE=False)."
    )

[OK] scene 00 | cysts=7 (expected 7)
[OK] scene 01 | cysts=7 (expected 7)
[OK] scene 02 | cysts=7 (expected 7)
[OK] scene 03 | cysts=7 (expected 7)
[OK] scene 04 | cysts=6 (expected 6)
[OK] scene 05 | cysts=7 (expected 7)
[OK] scene 06 | cysts=7 (expected 7)
[OK] scene 07 | cysts=6 (expected 6)
[OK] scene 08 | cysts=7 (expected 7)
[OK] scene 09 | cysts=7 (expected 7)
[OK] scene 10 | cysts=7 (expected 7)
[OK] scene 11 | cysts=7 (expected 7)
[OK] scene 12 | cysts=7 (expected 7)
[OK] scene 13 | cysts=7 (expected 7)
[OK] scene 14 | cysts=7 (expected 7)
[OK] scene 15 | cysts=7 (expected 7)
[OK] scene 16 | cysts=7 (expected 7)
[OK] scene 17 | cysts=7 (expected 7)
[OK] scene 18 | cysts=7 (expected 7)
[OK] scene 19 | cysts=7 (expected 7)
[OK] scene 20 | cysts=7 (expected 7)
[OK] scene 21 | cysts=7 (expected 7)
[OK] scene 22 | cysts=7 (expected 7)
[OK] scene 23 | cysts=7 (expected 7)
[OK] scene 24 | cysts=7 (expected 7)
[OK] scene 25 | cysts=7 (expected 7)
[OK] scene 26 | cysts=7 (expected 7)
[

In [ ]:
# -------------------------------
# Debug all scenes: verify cyst segmentation intermediates
# -------------------------------
fallback_scene_df = scene_stats_df.loc[
    scene_stats_df["threshold_variant"].astype(str) != "primary",
    [
        "scene_index",
        "scene_name",
        "threshold_method",
        "threshold_variant",
        "threshold_value",
        "autotune_used",
        "trim_applied",
    ],
].copy()

trimmed_scene_df = scene_stats_df.loc[
    scene_stats_df["trim_applied"].astype(bool),
    [
        "scene_index",
        "scene_name",
        "threshold_method",
        "threshold_variant",
        "threshold_value",
        "autotune_used",
        "trim_applied",
    ],
].copy()

if fallback_scene_df.empty:
    print("No scenes required an Otsu-scale fallback.")
else:
    print("Scenes that required an Otsu-scale fallback:")
    display(fallback_scene_df)

if trimmed_scene_df.empty:
    print("No scenes required primary-Otsu trimming.")
else:
    print("Scenes trimmed after primary Otsu to remove extra components:")
    display(trimmed_scene_df)

if RENDER_ALL_SCENE_SEGMENTATION_DEBUG:
    print(f"Rendering segmentation debug figures for {len(scene_manifest_df)} scenes.")
    for row in scene_manifest_df.itertuples(index=False):
        scene_idx = int(row.scene_index)
        channels_dbg, scene_shape_dbg, series0_shape_dbg, scene_name_dbg = (
            load_scene_channels(
                LOCATIONS_OME,
                scene_idx,
                CHANNEL_MAP,
                projection=PROJECTION,
            )
        )

        expected_dbg = expected_cyst_count_for_scene(scene_idx)
        cyst_params_dbg = dict(CYST_PARAMS)
        cyst_params_dbg["target_count"] = expected_dbg
        cyst_labels_dbg, cyst_props_dbg, cyst_debug_dbg = segment_cysts_from_dapi(
            channels_dbg["dapi"], cyst_params_dbg
        )

        plot_cyst_segmentation_debug_figure(
            channels=channels_dbg,
            cyst_labels=cyst_labels_dbg,
            cyst_props=cyst_props_dbg,
            cyst_debug=cyst_debug_dbg,
            scene_index=scene_idx,
            scene_name=scene_name_dbg,
            expected_cysts=expected_dbg,
        )
else:
    print(
        "Skipping per-scene segmentation debug figures (RENDER_ALL_SCENE_SEGMENTATION_DEBUG=False)."
    )

No scenes required an Otsu-scale fallback.
Scenes trimmed after primary Otsu to remove extra components:


    scene_index                 scene_name threshold_method threshold_variant  \
29           29  well3-36locations.czi #30             otsu           primary   

    threshold_value  autotune_used  trim_applied  
29       430.470703          False          True  

Skipping per-scene segmentation debug figures (RENDER_ALL_SCENE_SEGMENTATION_DEBUG=False).


## Next Step

Run `<analysis-root>/pSMAD/notebooks/02c_manual_bead_well_annotation.ipynb`.

That notebook reads:
- `results/tables/scene_manifest_36locations.tsv`
- `results/manual_bead_well_36locations/scene_images/`
- `results/manual_bead_well_36locations/labels/`

and then writes:
- `results/tables/manual_bead_well_positions_36locations.tsv`
- `results/tables/manual_roi_summary_36locations.tsv`
- `results/rois/manual_bead_well_36locations/*.json`